# 🔴 Solution: Mamba SSM Step

Selective state-space model with input-dependent B, C, and delta

In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def mamba_ssm_step(x, h, A, B, C, delta):
    """
    Mamba 选择性状态空间模型 (SSM) 的单步递推。

    核心创新：B、C 和 delta 是输入相关的（选择性的），
    不同于经典 SSM 中的固定参数。这使模型能够选择性地记忆或遗忘信息。

    参数:
        x: 当前步输入, shape = (B, D)
            B = batch size, D = 通道数（state 维度与 D 相同）
        h: 隐藏状态, shape = (B, D, N)
            N 是状态维度，每个通道有 N 个状态分量
        A: 状态转移矩阵, shape = (D, N)
            上游使用 A = -exp(A_log) 保证负值（稳定性）
        B: 输入投影（选择性）, shape = (B, D, N)
            与 x 相关，决定哪些信息被写入状态
        C: 输出投影（选择性）, shape = (B, D, N)
            与 x 相关，决定从状态中读取哪些信息
        delta: 离散化步长（选择性，经过 softplus）, shape = (B, D)
            控制每个通道的「遗忘速度」

    返回: (y, h_new)
        y: 输出, shape = (B, D)
        h_new: 更新后的隐藏状态, shape = (B, D, N)
    """
    # ---- Step 1: 零阶保持离散化 ----
    # 连续 SSM: dh/dt = Ah + Bx
    # 离散化 (零阶保持): h_{t+1} = exp(A*delta) * h_t + (A^{-1}(exp(A*delta)-I)) * B * x
    # 简化版本: dA = exp(delta * A), dB = delta * B

    # dA = exp(delta * A)
    # delta: (B, D) → unsqueeze(-1) → (B, D, 1)
    # A: (D, N) → unsqueeze(0) → (1, D, N)
    # 广播: (B, D, 1) * (1, D, N) = (B, D, N)
    # dA[b, d, n] = exp(delta[b, d] * A[d, n])
    dA = torch.exp(delta.unsqueeze(-1) * A.unsqueeze(0))  # (B, D, N)

    # dB = delta * B
    # delta: (B, D, 1), B: (B, D, N) → 广播 → (B, D, N)
    dB = delta.unsqueeze(-1) * B  # (B, D, N)

    # ---- Step 2: 状态更新 (递推) ----
    # h_new = dA * h + dB * x
    # dA * h: (B, D, N) * (B, D, N) = (B, D, N)  逐元素衰减旧状态
    # dB * x: (B, D, N) * (B, D, 1) = (B, D, N)  写入新信息
    # x.unsqueeze(-1): (B, D) → (B, D, 1)，广播到 N 个状态分量
    h_new = dA * h + dB * x.unsqueeze(-1)  # (B, D, N)

    # ---- Step 3: 输出 ----
    # y = Σ_n (h_new * C)[d, n]
    # h_new * C: (B, D, N) * (B, D, N) = (B, D, N)  逐元素读取
    # .sum(dim=-1): 在 N 维求和 → (B, D)
    # 直觉: C 决定从 N 个状态分量中读取多少，求和得到标量输出
    y = (h_new * C).sum(dim=-1)  # (B, D)

    return y, h_new

In [ ]:
torch.manual_seed(0)
B_size, D, N = 2, 8, 4  # batch, channels, state_dim

x     = torch.randn(B_size, D)
h     = torch.zeros(B_size, D, N)
A     = -torch.rand(D, N)          # negative for stability
B_mat = torch.randn(D, N)
C     = torch.randn(B_size, D, N)
delta = torch.rand(B_size, D).add(0.1)  # positive step sizes

y, h_new = mamba_ssm_step(x, h, A, B_mat, C, delta)

print("y shape:    ", y.shape)      # (2, 8)
print("h_new shape:", h_new.shape)  # (2, 8, 4)

# Verify dA = exp(delta * A) numerically for first element
dA_manual = torch.exp(delta[0, 0] * A[0])
dA_check  = torch.exp(delta[0:1, 0:1] * A[0:1])[0, 0]
print("dA formula check (should be ~0):", (dA_manual - dA_check).abs().max().item())

In [ ]:
from torch_judge import check
check("mamba_ssm")

## 📝 核心思路总结

1. **选择性是 Mamba 的灵魂**：B、C、delta 都是输入相关的，模型可以根据当前输入决定「写入什么、读取什么、遗忘多快」。
2. **零阶保持离散化**：`dA = exp(δ*A)`，`dB = δ*B`，将连续 SSM 离散化；A<0 保证 dA<1（衰减），δ>0 保证离散化有效。
3. **递推公式**：`h_new = dA * h + dB * x`，dA 是衰减因子（遗忘），dB*x 是新信息（记忆），y = (h_new * C).sum(-1) 是选择性读取。
4. **并行化**：虽然形式上是递推，但 dA 的结构允许用并行扫描 (parallel scan) 高效计算，这是 Mamba 比 Transformer 快的关键。